# Goal

Validate our equations for E and H in MANCOVA.  These forms remove redundant computation across the different voxels within one region:


$$ N r H =r \bar{Y}^T (Q_1 Q_1^T) \bar{Y} $$

$$ N r E = \sum_v Y_v ^T Y_v - r \bar{Y}^T (Q_0 Q_0^T  + Q_1 Q_1^T) \bar{Y} $$

Surprising to note: E is invariate to covariates (it is constant whether there are no covariates or if all but the final column of X is a covariate)

E is traditionally comptued by projecting Y and X into a space orthgonal to all covariates.  In doing so, the $Q_0Q_0^T$ should be removed from the rightward term but added back in by the $\sum_v Y_v^T Y_v$ term on the left.  Weird

# Prep

In [ ]:
import numpy as np

from dataclasses import dataclass

@dataclass
class ProblemConfig:
    a: int
    b: int
    num_img: int
    num_vox: int
    noise_scale: int
    
    
def build_problem(pc: ProblemConfig, seed: int = 0):
    """ samples a random problem """
    rng = np.random.default_rng(seed=seed)
    beta = rng.standard_normal((pc.a, pc.b))
    x = rng.standard_normal((pc.num_img, pc.a))

    xr = np.tile(x, (pc.num_vox, 1))
    yr = xr @ beta + rng.standard_normal((pc.num_img * pc.num_vox, pc.b)) * pc.noise_scale
    
    return x, xr, yr

def covariate_project(xr, yr, a_prime):
    """ projects xr and yr into orthogonal space of first a_prime columns of xr """
    if not a_prime:
        # no covariates to remove
        return xr, yr
    
    # remove component of x and y in direction of covariates
    _xr = xr[:, :a_prime]
    p0 = np.eye(yr.shape[0]) - _xr @ np.linalg.pinv(_xr)
    _yr = p0 @ yr
    _xr = p0 @ xr  
        
    return _xr, _yr

# Validating H

In [ ]:
pc = ProblemConfig(a=4, b=2, num_img=11, num_vox=7, noise_scale=10)
seed = 0

# generate problem
x, xr, yr = build_problem(pc, seed=seed)

# fast (prep)
q, r = np.linalg.qr(x)
yr3d = np.stack([yr[i * pc.num_img: (i + 1) * pc.num_img, :] 
                 for i in range(pc.num_vox)], axis=2)
y_mean = yr3d.mean(axis=2)

# test for any meaningful number of covariates (expression doesn't change)
for a_prime in range(pc.a - 1):
    # fast
    q1 = q[:, a_prime:pc.a]
    H = pc.num_vox * y_mean.T @ q1 @ q1.T @ y_mean
    
    # control for covariates
    _xr, _yr = covariate_project(xr, yr, a_prime)

    # regress in this adjusted space
    # NOTE: must use _xr[:, a_prime] not _xr below to avoid sending a close-to-zero 
    # vector to pinv, doing so invites poor condition number and numerical instability
    p = _xr[:, a_prime:] @ np.linalg.pinv(_xr[:, a_prime:])
    yhat = p @ _yr
    H_exp = yhat.T @ yhat
        
    assert np.allclose(H, H_exp)

# Validating E

In [ ]:
pc = ProblemConfig(a=4, b=2, num_img=11, num_vox=7, noise_scale=10)
for seed in range(100):
    # generate problem
    x, xr, yr = build_problem(pc, seed=seed)
    
    # fast
    q, r = np.linalg.qr(x)
    yr3d = np.stack([yr[i * pc.num_img: (i + 1) * pc.num_img, :] 
                     for i in range(pc.num_vox)], axis=2)
    y_mean = yr3d.mean(axis=2)

    E = yr.T @ yr - pc.num_vox * y_mean.T @ q @ q.T @ y_mean
    
    # test for any meaningful number of covariates (expression doesn't change)
    for a_prime in range(pc.a - 1):
        # control for covariates
        _xr, _yr = covariate_project(xr, yr, a_prime)
        
        # regress in this adjusted space
        # NOTE: must use _xr[:, a_prime] not _xr below to avoid sending a close-to-zero 
        # vector to pinv, doing so invites poor condition number and numerical instability
        p = np.eye(_yr.shape[0]) - _xr[:, a_prime:] @ np.linalg.pinv(_xr[:, a_prime:])
        error = p @ _yr
        E_exp = error.T @ error

        assert np.allclose(E_exp, E)

# another version

In [ ]:
import numpy as np

a = 3
b = 2
num_img = 5
num_vox = 7

rng = np.random.default_rng(seed=0)
x = rng.standard_normal((a, num_img))
y3d = rng.standard_normal((b, num_img, num_vox))
y = y3d.reshape((b, -1), order='F')

# test: single voxel test projection as q.T @ q
q, r = np.linalg.qr(x.T)
q, r = q.T, r.T
p_trusted = x.T @ np.linalg.inv(x @ x.T) @ x
p_exp = q.T @ q
assert np.allclose(p_trusted, p_exp)

# test: multi vox q_multi = 1 / sqrt(r) [q, q, ...]
x_multi = np.kron(np.ones((1, num_vox)), x)
q_multi, r_multi = np.linalg.qr(x_multi.T)
q_multi, r_multi = q_multi.T, r_multi.T
_q_multi = np.kron(np.ones((1, num_vox)), q) / num_vox ** .5
assert np.allclose(_q_multi, q_multi)

# test: centering matrix for spatial cov
c = np.kron(np.ones((num_vox, 1)), np.eye(num_img)) / num_vox
y_mean = y3d.mean(axis=2)
assert np.allclose(y @ c, y_mean)

# test: spatial covariance compute
diff = y - np.kron(np.ones((1, num_vox)), y_mean)
space_cov_trusted = diff @ diff.T
space_cov = y @ (np.eye(num_vox * num_img) - num_vox * c @ c.T) @ y.T
assert np.allclose(space_cov_trusted, space_cov)

# test: H from QR factorization (space naive)
h_trusted = y @ x_multi.T @ np.linalg.inv(x_multi @ x_multi.T) @ x_multi @ y.T
h0 = y @ q_multi.T @ q_multi @ y.T
assert np.allclose(h0, h_trusted)

# test: H from QR factorization (quick compute across space)
h1 = num_vox * y_mean @ q.T @ q @ y_mean.T
assert np.allclose(h1, h0)

# test: e
e = space_cov + num_vox * y_mean @ (np.eye(num_img) - q.T @ q) @ y_mean.T
p = np.eye(num_img * num_vox) - np.linalg.pinv(x_multi) @ x_multi
e_trusted = y @ p @ y.T
assert np.allclose(e, e_trusted)

# test: control for covariates still works (WLOG assume first few rows are
# covariates while remaining rows of x are "of interest").
# to ensure orthogonality to covariates swap in q1 for all q's above
for a_prime in range(a + 1):
    q0 = q[:a_prime, :]
    q1 = q[a_prime:, :]
    np.allclose(q.T @ q, q0.T @ q0 + q1.T @ q1)
